In [17]:
import numpy as np
import gensim
import pandas as pd
from scipy import spatial

In [6]:
def cosine_similarity(vec1, vec2):
    return 1.0 - spatial.distance.cosine(vec1, vec2)

In [12]:
def get_avg_vec(emb, word, k):
    tmp = []
    tmp.append(emb[word])
    for closest_word, similarity in emb.most_similar(positive=word, topn=k):
        tmp.append(emb[closest_word])
    avg_vec = np.mean(tmp, axis=0)
    return avg_vec

In [13]:
def transform_antonym_to_axis(emb, antonym, k):
    if k == 0:
        return emb[antonym[1]] - emb[antonym[0]]

    else:
        vec_antonym_1 = get_avg_vec(emb, antonym[1], k)
        vec_antonym_0 = get_avg_vec(emb, antonym[0], k)

        return vec_antonym_1 - vec_antonym_0 

In [14]:
def project_word_on_axis(emb, word, antonym, k=10):
    return cosine_similarity(emb[word], transform_antonym_to_axis(emb, antonym, k)) 

In [ ]:
EMBEDDING_PATH = "~/SemAxis" ### Yout path to embedding file 

######################################################################
### Google News embedding (Download: https://code.google.com/archive/p/word2vec/). Note that for SemAxis, bin file needs to be converted to text file: see https://stackoverflow.com/questions/27324292/convert-word2vec-bin-file-to-text)

# test_path = "%s/GoogleNews-vectors-negative300.txt" % (EMBEDDING_PATH)
######################################################################

######################################################################
## Reddit20M embedding (Download: https://drive.google.com/file/d/1ewmS5Uu4tWAkwWsuY8FZVgLr85vvZXye/view?usp=sharing) 

test_path = "%s/Reddit20M.cbow.300.100.txt" % (EMBEDDING_PATH)
######################################################################

embeddings = gensim.models.KeyedVectors.load_word2vec_format(test_path)

In [31]:
semantic_axes = pd.read_csv('{}/732_semaxis_axes.tsv'.format(EMBEDDING_PATH), sep='\t', header=None)

In [32]:
semantic_axes

,0,1
0,uneatable,eatable
1,inapplicable,applicable
2,emetic,antiemetic
3,nonsynonymous,synonymous
4,infernal,supernal
...,...,...
727,overpriced,underpriced
728,frequently,infrequently
729,apathetic,empathic
730,divided,undivided


In [36]:
semantic_axes[1][1]

'applicable'

In [37]:
def semaxis(embeddings, word, i, k=3):
    print(semantic_axes[0][i], semantic_axes[1][i])
    return project_word_on_axis(embeddings, "baby", (semantic_axes[0][i], semantic_axes[1][i]), k=3)

In [62]:
i = 54
print(semaxis(embeddings, "baby", i, k=3))

## Test results (with k=3) should be: 
## 0.16963434219360352 with Google News embedding
## 0.31472429633140564 with Reddit20M embedding

false real
0.10155487805604935


In [69]:
print(project_word_on_axis(test_embed, "baby", ("ugly", "adorable"), k=3))
print(project_word_on_axis(test_embed, "baby", ("silent", "noisy"), k=3))
print(project_word_on_axis(test_embed, "baby", ("young", "old"), k=3))
print(project_word_on_axis(test_embed, "baby", ("dangerous", "harmless"), k=3))

0.1621214896440506
0.04698548838496208
-0.03787761926651001
0.03690413013100624
